In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transformSales_person(SalesPerson_df):

    windowSpec_rw = Window.partitionBy("BusinessEntityID").orderBy("BusinessEntityID")
    SalesPerson_df = SalesPerson_df.withColumn("MiddleName",F.when(F.col("MiddleName").isNull(), F.lit("")).otherwise(F.col("MiddleName"))).withColumn("PersonType ",F.when(~(F.col("PersonType").isin(['ÉM','SP'])), F.lit("OT")).otherwise(F.col("MiddleName"))).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    SalesPerson_df = SalesPerson_df.filter(F.col("rw") == 1).drop("rw")
    SalesPerson_df = SalesPerson_df.withColumn("processed_timestamp", F.current_timestamp())
    SalesPerson_df = SalesPerson_df.select( 
         F.col("BusinessEntityID").cast(IntegerType()).alias("BusinessEntityID"),
         F.col("AddressID").cast(IntegerType()).alias("AddressID"),
         F.col("AddressTypeID").cast(IntegerType()).alias("AddressTypeID"),
         F.col("rowguid").cast(StringType()).alias("rowguid"),
         F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
         F.col("AddressLine1").cast(StringType()).alias("AddressLine1"),
         F.col("AddressLine2").cast(StringType()).alias("AddressLine2"),
         F.col("City").cast(StringType()).alias("City"),
         F.col("StateProvinceID").cast(IntegerType()).alias("StateProvinceID"),
         F.col("PostalCode").cast(StringType()).alias("PostalCode"),
         F.col("SpatialLocation").cast(StringType()).alias("SpatialLocation"),
         F.col("CountryRegionCode").cast(StringType()).alias("CountryRegionCode"),
         F.col("Name").cast(StringType()).alias("Name"),
         F.col("PersonID").cast(IntegerType()).alias("PersonID"),
         F.col("ContactTypeID").cast(IntegerType()).alias("ContactTypeID"),
         F.col("PasswordHash").cast(StringType()).alias("PasswordHash"),
         F.col("PasswordSalt").cast(StringType()).alias("PasswordSalt"),
         F.col("StateProvinceCode").cast(StringType()).alias("StateProvinceCode"),
         F.col("IsOnlyStateProvinceFlag").cast(BooleanType()).alias("IsOnlyStateProvinceFlag"),
         F.col("PhoneNumber").cast(StringType()).alias("PhoneNumber"),
         F.col("EmailAddress").cast(StringType()).alias("EmailAddress"),
         F.col("PersonType").cast(StringType()).alias("PersonType"),
         F.col("NameStyle").cast(BooleanType()).alias("NameStyle"),
         F.col("Title").cast(StringType()).alias("Title"),
         F.trim(F.col("FirstName")).cast(StringType()),
         F.trim(F.col("MiddleName")).cast(StringType()),
         F.trim(F.col("LastName")).cast(StringType()),
         F.trim(F.col("Suffix")).cast(StringType()),
         F.col("EmailPromotion").cast(IntegerType()).alias("EmailPromotion"),
         F.col("AdditionalContactInfo").cast(StringType()),
         F.col("Demographics").cast(StringType()),
         F.col("_rescued_data").cast(StringType()).alias("_rescued_data"),
         F.col("processed_timestamp").cast(DateType()).alias("processed_timestamp"))
                                 
    return SalesPerson_df




if __name__ == "__main__":

    SalesPerson_tbl = dbutils.widgets.get("SalesOrderPerson")
    SalesPerson_df = df = spark.read.table(SalesPerson_tbl)
    SalesPerson_df_tgt = transformSales_person(SalesPerson_df)
    display(SalesPerson_df_tgt)